# Embeddingi i wektorowa baza danych – zadania

## Embeddingi

**Funkcje pomocnicze** – uruchom poniższą komórkę, aby korzystać z funkcji do wyszukiwania najbardziej podobnych tekstów.

- `most_similar()` zwraca listę par `(podobieństwo, dokument)` posortowaną malejąco,
- `print_most_similar()` wyświetla ranking.

Obie działają z dowolnym obiektem `embeddings` z LangChaina. Opcjonalnie: `top_k` – liczba wyników, `document_vectors` – gotowe wektory dokumentów (żeby nie liczyć ich od nowa).

In [ ]:
import numpy as np


def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def most_similar(query, documents, embeddings, top_k=None, document_vectors=None):
    """Zwraca listę par (podobieństwo, dokument) posortowaną malejąco."""
    query_vector = embeddings.embed_query(query)

    if document_vectors is None:
        document_vectors = embeddings.embed_documents(documents)

    similarities = [cosine_similarity(query_vector, vector) for vector in document_vectors]

    return sorted(zip(similarities, documents), reverse=True)[:top_k]


def print_most_similar(query, documents, embeddings, top_k=None, document_vectors=None):
    """Wyświetla ranking dokumentów najbardziej podobnych do zapytania."""
    results = most_similar(query, documents, embeddings, top_k, document_vectors)

    print(f"Zapytanie: {query}\n---")
    for i, (similarity, document) in enumerate(results, 1):
        print(f"{i}. [{similarity:.4f}] {document}\n")

1. Wykonaj ponownie podział pierwszej księgi "Pana Tadeusza" na chunki za pomocą dowolnej metody. Następnie stwórz embeddingi chunków używając wybranego modelu z HuggingFace.

Napisz prompta nawiązującego do dowolnego fragmentu z tego tekstu i znajdź najbardziej podobne chunki. Użyj w tym celu funkcji `print_most_similar()` z komórki powyżej.

Propozycje:
- `"Litwo, ojczyzno moja"`
- `"Spór sąsiedzki"`
- `"Moskiewski kapitan"`

---

Rekomendowany model to `sdadas/mmlw-roberta-base`, ale jeśli zdążysz to wypróbuj również pozostałe modele. Pamiętaj, że:
- model `mmlw-roberta-base` wymaga prefiksu `"zapytanie: "` przed zapytaniem – najwygodniej ustawić go przez `query_encode_kwargs={"prompt": "zapytanie: "}`,
- każdy model ma limit tokenów (`max_seq_length`) – dłuższe chunki zostaną ucięte,
- tokeny modelu do embeddingów nie muszą pokrywać się z tokenami, które liczy `TokenTextSplitter` (tiktoken).

---
(czas: 16 min.)

In [ ]:
# ...

---
---
---

2. Zwektoryzuj całość "Pana Tadeusza" z pliku `documents/pan-tadeusz.pdf`, a następnie sprawdź, które chunki są najbardziej podobne do pytań z listy `questions` poniżej. Wyświetl *k* najbardziej pasujących chunków, np. *n=k* (parametr `top_k`).

Porównaj różne modele. Pamiętaj, że tworzenie się embeddingów może potrwać parę minut. Wykorzystaj ten czas na pisanie dalszej części kodu.

**Wskazówka:** zembeduj chunki tylko raz, a gotowe wektory przekazuj do `print_most_similar()` w parametrze `document_vectors` – dzięki temu nie będą liczone od nowa przy każdym pytaniu.

Przejrzyj wyniki dla poszczególnych pytań i zastanów się, czy wyszukiwanie podobnych fragmentów działa poprawnie.

---
(czas: 18 min.)

In [ ]:
questions = [
    "Pomiędzy kim trwa spór o zamek Horeszków?",
    "Jak miał na imię kapitan Ryków?",
    "Wymień wszystkie zioła które znajdowały się w donicach na oknach w domu Tadeusza.",
    "'Kapitanie! nie splamisz czci twojej pardonem. Dałeś próby, rycerzu nieszczęsny, lecz mężny' – kto wypowiada te słowa?",
    "Od kogo Sędzia otrzymał w darze kryształowy kufel?",
    "Czym się różnili Dobrzyńscy od reszty Litwinów?",
    "Co mówi Sędzia o wódce gdańskiej?",
    "Jak Domejko i Dowejko nazwali karczmę którą zbudowali po zakończeniu sporu?",
    "Kto wypowiedział słowa 'Kometa czasem wojny, czasem wróży kłótnie!'",
    "Jakie ustawy znajdują się w żółtej księdze?"
]

In [ ]:
# ...

## Qdrant

1. Stwórz kolekcję `pan_tadeusz` i dodaj do niej zembeddowane chunki całego "Pana Tadeusza".

**Kroki:**
1. Wczytaj **osobno każdy z 13 plików PDF** z folderu `documents/pan-tadeusz/` (12 ksiąg i epilog) za pomocą `PyMuPDFLoader` – nie korzystaj z pliku `documents/pan-tadeusz.pdf` z całością.
2. Podziel dokumenty na chunki za pomocą `RecursiveCharacterTextSplitter` z separatorami `"\n\n"`, `"\n"`, `". "` i metodą `split_documents()`. Dobierz `chunk_size` i `chunk_overlap` (np. 1000 i 200).
3. Stwórz embeddingi chunków (rekomendowany model: `sdadas/mmlw-roberta-base`).
4. Utwórz kolekcję z miarą `Distance.COSINE`. Rozmiar wektora odczytaj z modelu zamiast wpisywać go na sztywno.
5. Dodaj punkty do kolekcji za pomocą `client.upsert()` i `PointStruct`. W payloadzie umieść:
   - `page_content` – treść chunka,
   - `metadata` – metadane chunka

   Przy czym metadane powinny obejmować:
   - `chunk_length` – liczba znaków w chunku,
   - `book_number` – numer księgi jako `int`, wzięty z początku nazwy pliku (np. `9` dla `09_księga_dziewiąta.pdf`); dla epilogu wartość `None`,
   - `title` – tytuł księgi (z metadanych `PyMuPDFLoader`),
   - `page` – numer strony,
   - `model` – nazwa modelu, którego użyto do stworzenia embeddingu.

**Oczekiwany wynik:** punkty w kolekcji z payloadem podobnym do:

```python
{
    "page_content": "Księga pierwsza – Gospodarstwo\nPowrót panicza — Spotkanie się pierwsze w pokoiku, ...",
    "metadata": {
     "chunk_length": 964,
     "book_number": 1,
     "title": "Księga pierwsza – Gospodarstwo",
     "page": 0,
     "model": "sdadas/mmlw-roberta-base"   
    }
}
```

Sprawdź w dashboardzie Qdranta (http://localhost:6333/dashboard#/collections), czy punkty zostały dodane.

---
(czas: 18 min.)

In [ ]:
# ...

## Qdrant w LangChain

1. Usuń i utwórz ponownie pustą kolekcję `pan_tadeusz` i wypełnij ją wszystkimi chunkami. Tym razem zrób to korzystając z LangChaina – klasy `QdrantVectorStore` i metody **`add_documents()`**.

Chunki z poprzedniego zadania są już obiektami `Document`, więc wystarczy przygotować dla nich odpowiednie `metadata` i przekazać je do `add_documents()` – embeddingi zostaną policzone automatycznie. W `metadata` umieść: `chunk_length`, `book_number`, `title`, `page` oraz `model`.

Zwróć uwagę na różnicę w kolejności chunków, id punktów oraz strukturze payloadu w porównaniu z poprzednim zadaniem.

---
(czas: 10 min.)

In [ ]:
# ...